In [ ]:
from spint import Gravity, Production, Attraction, Doubly
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr
import src._plot_utils as pu

jobs = pd.read_csv("data/data4report/lookup_census.csv", index_col=1)

jobs = pd.read_csv("data/data4report/lads_jobs_2021.csv", index_col=1)
pop = pd.read_csv("data/data4report/lads_jobs_pop_2021.csv", index_col=0)
pop.index = pop["Area code"]

m_od = pd.read_csv("data/data4report/m_od.csv", index_col=0)
c_od = pd.read_csv("data/data4report/c_od.csv", index_col=0)

distance_df = pd.read_csv("data/data4report/dist.csv")
distance_df.set_index(["from", "to"], inplace=True)


def process_od(od, distance_df, pop, jobs):
    dij = []
    flows = []
    Origin = []
    Destination = []
    Oi = []
    Dj = []

    for i in od.columns.values:
        for j in od.index.values:
            if i != j:
                flows.append(od.loc[i, j])
                dij.append(distance_df.loc[(i, j), "distance"])
                Origin.append(i)
                Destination.append(j)
                if i in pop.index:
                    Oi.append(int(pop.loc[i]["In employment 2021"].replace(",", "")))
                else:
                    Oi.append(0.1)

                if j in jobs.index:
                    Dj.append(int(jobs.loc[j]["Total jobs"].replace(",", "")))
                else:
                    Dj.append(0.1)

    dij = np.array(dij)
    flows = np.array(flows)
    Oi = np.array(Oi)
    Dj = np.array(Dj)
    dij = np.array(dij)
    Origin = np.array(Origin)
    Destination = np.array(Destination)

    return dij, flows, Origin, Destination, Oi, Dj


def calculate_params(flows, Oi, Dj, dij, Origin=None, Destination=None, func="pow"):
    grav = Gravity(flows, Oi, Dj, dij, cost_func=func)
    prod = Production(flows, Origin, Dj, dij, cost_func=func)
    attr = Attraction(flows, Destination, Oi, dij, cost_func=func)
    doub = Doubly(flows, Origin, Destination, dij, cost_func=func)

    return grav, prod, attr, doub


def process_models(models):
    R2, adjR2, SSI, SRMSE, AIC = [], [], [], [], []
    model_name = ["Gravity", "Production", "Attraction", "Doubly"]

    for model in models:
        R2.append(round(model.pseudoR2, 2))
        adjR2.append(round(model.adj_pseudoR2, 2))
        SSI.append(round(model.SSI, 2))
        SRMSE.append(round(model.SRMSE, 2))
        AIC.append(round(model.AIC, 2))

    cols = {
        "model_name": model_name,
        "R2": R2,
        "adjR2": adjR2,
        "SSI": SSI,
        "SRMSE": SRMSE,
        "AIC": AIC,
    }

    data = pd.DataFrame(cols).set_index("model_name")
    return data


def process_tvalues(models):
    model_names = np.array(["Gravity", "Production", "Attraction", "Doubly"]).reshape(
        -1, 1
    )

    params1 = np.round(models[0].tvalues[-4:].T, 3)
    params2 = np.round(models[1].tvalues[-4:].T, 3)
    params3 = np.round(models[2].tvalues[-4:].T, 3)
    params4 = np.round(models[3].tvalues[-4:].T, 3)

    result = np.vstack((params1, params2, params3, params4))
    return np.hstack((model_names, result))

In [ ]:
dij, flows, Origin, Destination, Oi, Dj = process_od(c_od, distance_df, pop, jobs)

func = "exp"  # "pow" or "exp"
grav, prod, attr, doub = calculate_params(
    flows, Oi, Dj, dij, Origin, Destination, func=func
)

data = process_models([grav, prod, attr, doub])

In [ ]:
dij, flows, Origin, Destination, Oi, Dj = process_od(m_od, distance_df, pop, jobs)

func = "exp"  # "pow" or "exp"
grav, prod, attr, doub = calculate_params(
    flows, Oi, Dj, dij, Origin, Destination, func=func
)

data = process_models([grav, prod, attr, doub])

In [ ]:
c_od.to_numpy().flatten()

In [ ]:
fig, axs = plt.subplots(
    1, 2, figsize=(18 * pu.cm, 8 * pu.cm), layout="constrained", dpi=300
)

flat_c = c_od.to_numpy().flatten()
flat_m = m_od.to_numpy().flatten()
mask = np.isfinite(flat_c) & np.isfinite(flat_m)
flat_c = flat_c[mask]
flat_m = flat_m[mask]

log_c = np.log1p(flat_c)
log_m = np.log1p(flat_m)
raw_corr = np.corrcoef(flat_c, flat_m)[0, 1]
# log_corr = np.corrcoef(log_c, log_m)[0, 1]


def func(x, a):
    return a * x


from scipy.optimize import curve_fit

popt, pcov = curve_fit(func, flat_c, flat_m)
x_fit = np.linspace(min(flat_c), max(flat_c), 100000)
r2 = pearsonr(flat_c, flat_m)[0] ** 2

axs[0].plot(
    np.log1p(x_fit),
    np.log1p(func(x_fit, *popt)),
    color="red",
    label=f"y={popt[0]:.4f}x, $R^2$={r2:.2f}",
)
axs[0].scatter(np.log1p(flat_c), np.log1p(flat_m), s=1, alpha=1, edgecolors="k")
axs[0].set_xlabel("Log1p Census OD flows")
axs[0].set_ylabel("Log1p Mobile OD flows")
# axs[0].text(0.05, 0.95, f"r = {raw_corr:.2f}", transform=axs[0].transAxes, ha='left', va='top')
axs[0].legend(frameon=False, loc="upper left")
axs[0].set_xlim(-1, 12)
# axs[1].scatter(log_c, log_m, s=1, alpha=1, edgecolors='k')
# axs[1].set_ylim(-1,10)
# axs[1].set_xlabel('Log1p Census OD flows')
# axs[1].set_ylabel('Log1p Modelled OD flows')
# # axs[1].text(0.05, 0.95, f"r = {log_corr:.2f}", transform=axs[1].transAxes, ha='left', va='top')
# axs[1].plot(np.log1p(x_fit), np.log1p(func(x_fit, *popt)), color='red', label=f'y={popt[0]:.4f}x, $R^2$={r2:.2f}')
# axs[1].legend(frameon=False, loc='upper left')

mask = (flat_c > 0) & (flat_m > 0)
flat_c_f = flat_c[mask]
flat_m_f = flat_m[mask]

popt, pcov = curve_fit(func, flat_c_f, flat_m_f)
x_fit = np.linspace(0, max(flat_c_f), 100000)
r2 = pearsonr(flat_c_f, flat_m_f)[0] ** 2

axs[1].plot(
    np.log1p(x_fit),
    np.log1p(func(x_fit, *popt)),
    color="red",
    label=f"y={popt[0]:.4f}x, $R^2$={r2:.2f}",
)
axs[1].scatter(np.log1p(flat_c_f), np.log1p(flat_m_f), s=1, alpha=1, edgecolors="k")
axs[1].set_xlabel("Log1p Census OD flows")
axs[1].set_ylabel("Log1p Mobile OD flows")
# axs[0].text(0.05, 0.95, f"r = {raw_corr:.2f}", transform=axs[0].transAxes, ha='left', va='top')
axs[1].legend(frameon=False, loc="upper left")
axs[1].set_xlim(-1, 12)

fig.savefig("fig/od_scatter_census_mobile.pdf", dpi=300)

In [ ]:
(
    ((flat_c > 0) & (flat_m == 0)).sum(),
    ((flat_c == 0) & (flat_m == 0)).sum(),
    ((flat_c == 0) & (flat_m > 0)).sum(),
    ((flat_c > 0) & (flat_m > 0)).sum(),
)

In [ ]:
mask  # 31963 / 95481